# TP Spark 3 - MLlib avec MNIST (Decision Tree et Random Forest)

Objectifs du notebook :
- charger MNIST depuis une URL ;
- preparer les donnees pour Spark MLlib ;
- faire un train/test split ;
- entrainer un arbre de decision puis une foret aleatoire ;
- comparer rapidement les temps d'entrainement Spark vs scikit-learn ;
- proposer un exercice de comparaison reproductible sur vos machines.

In [1]:
import os
import time
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from sklearn.ensemble import RandomForestClassifier as SkRandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

spark = SparkSession.builder.appName("TP_MLlib_MNIST").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark pret. Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/20 11:30:17 WARN Utils: Your hostname, codespaces-e1abfc, resolves to a loopback address: 127.0.0.1; using 10.0.0.216 instead (on interface eth0)
26/03/20 11:30:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 11:30:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark pret. Version : 4.1.1


## 1. Telecharger MNIST

Le jeu de donnees est compose de 2 fichiers CSV :
- `mnist_train.csv`
- `mnist_test.csv`

Format :
- colonne 1 = label (digit 0 a 9) ;
- colonnes suivantes = pixels (0..255).

In [3]:
url = "https://www.math.univ-toulouse.fr/~besse/Wikistat/data/"
data_dir = "Data/mnist"
os.makedirs(data_dir, exist_ok=True)

for filename in ["mnist_train.csv", "mnist_test.csv"]:
    target_path = os.path.join(data_dir, filename)
    if not os.path.exists(target_path):
        print(f"Telechargement de {filename}...")
        urllib.request.urlretrieve(url + filename, target_path)
    else:
        print(f"Fichier deja present : {target_path}")

print("Telechargement termine.")

Fichier deja present : Data/mnist/mnist_train.csv
Fichier deja present : Data/mnist/mnist_test.csv
Telechargement termine.


## 2. Charger et inspecter les donnees

On affiche quelques informations utiles pour la prise en main :
- taille d'echantillon ;
- nombre de colonnes ;
- digits presents ;
- repartition par classe.

In [4]:
train_path = os.path.join(data_dir, "mnist_train.csv")
test_path = os.path.join(data_dir, "mnist_test.csv")

df_train_raw = spark.read.option("header", False).option("inferSchema", True).csv(train_path)
df_test_raw = spark.read.option("header", False).option("inferSchema", True).csv(test_path)
df_all_raw = df_train_raw.unionByName(df_test_raw)

print("Lignes train source :", df_train_raw.count())
print("Lignes test source  :", df_test_raw.count())
print("Lignes total        :", df_all_raw.count())
print("Nombre de colonnes  :", len(df_all_raw.columns))

digits = [int(r[0]) for r in df_all_raw.select("_c0").distinct().orderBy("_c0").collect()]
print("Digits detectes :", digits)

print("Repartition des labels :")
df_all_raw.groupBy("_c0").count().orderBy("_c0").show()

df_all_raw.show(3, truncate=False)

Lignes train source : 60000
Lignes test source  : 10000
Lignes total        : 70000
Nombre de colonnes  : 785


Digits detectes : [0]
Repartition des labels :


+---+-----+
|_c0|count|
+---+-----+
|  0|70000|
+---+-----+



26/03/20 11:31:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----

## 3. Preparation MLlib

On renomme :
- `_c0` en `label`
- `_c1` ... `_c784` en `px_1` ... `px_784`

Puis on cree la colonne `features` (vecteur dense) avec `VectorAssembler`.

In [5]:
pixel_cols = [f"_c{i}" for i in range(1, 785)]
rename_expr = [F.col("_c0").cast("double").alias("label")] + [
    F.col(c).cast("double").alias(f"px_{i}")
    for i, c in enumerate(pixel_cols, start=1)
]

df_mnist = df_all_raw.select(*rename_expr)
feature_cols = [f"px_{i}" for i in range(1, 785)]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df_features = assembler.transform(df_mnist).select("label", "features")
print("Schema pour ML :")
df_features.printSchema()

print("Apercu label/features :")
df_features.show(2, truncate=False)

Schema pour ML :
root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)

Apercu label/features :


26/03/20 11:31:41 WARN DAGScheduler: Broadcasting large task binary with size 1245.1 KiB


+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 4. Train / Test split

On fait ici un split aleatoire 80/20 pour rester dans un workflow standard de ML supervisé.

In [6]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

n_train = train_df.count()
n_test = test_df.count()
n_total = n_train + n_test

print("Taille train :", n_train)
print("Taille test  :", n_test)
print("Taille totale: " + str(n_total) + f" (train={n_train/n_total:.2%}, test={n_test/n_total:.2%})")

26/03/20 11:31:43 WARN DAGScheduler: Broadcasting large task binary with size 1274.2 KiB
26/03/20 11:31:59 WARN DAGScheduler: Broadcasting large task binary with size 1274.2 KiB


Taille train : 56232
Taille test  : 13768
Taille totale: 70000 (train=80.33%, test=19.67%)


## 5. Modele 1 - Decision Tree (Spark MLlib)

In [6]:
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=10, seed=42)

t0 = time.perf_counter()
dt_model = dt.fit(train_df)
dt_train_time = time.perf_counter() - t0

dt_pred = dt_model.transform(test_df)

acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

dt_acc = acc_eval.evaluate(dt_pred)
dt_f1 = f1_eval.evaluate(dt_pred)

print(f"Decision Tree - temps entrainement : {dt_train_time:.2f} s")
print(f"Decision Tree - accuracy          : {dt_acc:.4f}")
print(f"Decision Tree - f1-score          : {dt_f1:.4f}")

26/03/20 11:03:57 WARN DAGScheduler: Broadcasting large task binary with size 1288.3 KiB
26/03/20 11:04:05 WARN DAGScheduler: Broadcasting large task binary with size 1290.7 KiB
26/03/20 11:04:08 WARN DAGScheduler: Broadcasting large task binary with size 1290.5 KiB
26/03/20 11:04:16 WARN DAGScheduler: Broadcasting large task binary with size 1302.5 KiB
26/03/20 11:04:24 WARN DAGScheduler: Broadcasting large task binary with size 1445.1 KiB
26/03/20 11:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1306.5 KiB
26/03/20 11:04:40 WARN DAGScheduler: Broadcasting large task binary with size 1306.5 KiB


Decision Tree - temps entrainement : 36.65 s
Decision Tree - accuracy          : 1.0000
Decision Tree - f1-score          : 1.0000


## 6. Modele 2 - Random Forest (Spark MLlib)

In [7]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=40,
    maxDepth=12,
    seed=42
)

t0 = time.perf_counter()
rf_model = rf.fit(train_df)
rf_train_time = time.perf_counter() - t0

rf_pred = rf_model.transform(test_df)
rf_acc = acc_eval.evaluate(rf_pred)
rf_f1 = f1_eval.evaluate(rf_pred)

print(f"Random Forest (Spark) - temps entrainement : {rf_train_time:.2f} s")
print(f"Random Forest (Spark) - accuracy          : {rf_acc:.4f}")
print(f"Random Forest (Spark) - f1-score          : {rf_f1:.4f}")

26/03/20 11:04:46 WARN DAGScheduler: Broadcasting large task binary with size 1288.3 KiB
26/03/20 11:04:53 WARN DAGScheduler: Broadcasting large task binary with size 1290.8 KiB
26/03/20 11:04:55 WARN DAGScheduler: Broadcasting large task binary with size 1290.6 KiB
26/03/20 11:05:02 WARN DAGScheduler: Broadcasting large task binary with size 1302.5 KiB
26/03/20 11:05:09 WARN DAGScheduler: Broadcasting large task binary with size 1452.9 KiB
26/03/20 11:05:19 WARN DAGScheduler: Broadcasting large task binary with size 1401.8 KiB
26/03/20 11:05:26 WARN DAGScheduler: Broadcasting large task binary with size 1401.8 KiB


Random Forest (Spark) - temps entrainement : 33.55 s
Random Forest (Spark) - accuracy          : 1.0000
Random Forest (Spark) - f1-score          : 1.0000


## 7. Exercice  - Benchmark Spark (Codespace)

Objectif : mesurer le temps d'entrainement d'un Random Forest exigeant sur Spark dans Codespace.

Consignes (Spark uniquement) :
1. Dupliquer le dataset MNIST **5 fois**.
2. Faire un train/test split.
3. Entrainer un Random Forest exigeant (`numTrees=300`, `maxDepth=20`, `maxBins=64`).
4. Mesurer uniquement le temps de `fit()`.
5. Faire 3 runs et garder la mediane.

Dans la section 8, vous ferez **la meme chose en local avec scikit-learn** pour comparaison indicative.

In [7]:
# Exercice Spark (Codespace): duplication x5 + Random Forest exigeant

import time
import numpy as np
from pyspark.ml.classification import RandomForestClassifier

# 1) Dupliquer le dataset x5
factor = 5
df_x5 = df_features
for _ in range(factor - 1):
    df_x5 = df_x5.unionByName(df_features)

# Cache pour stabiliser les timings
df_x5 = df_x5.cache()
_ = df_x5.count()

# 2) Split
train_x5, test_x5 = df_x5.randomSplit([0.8, 0.2], seed=42)
print("Taille train Spark x5 :", train_x5.count())
print("Taille test  Spark x5 :", test_x5.count())

# 3) Modele exigeant
rf_spark_hard = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=300,
    maxDepth=20,
    maxBins=64,
    seed=42,
)

# 4) Mesurer le fit uniquement (3 runs)
spark_fit_times = []
for i in range(3):
    t0 = time.perf_counter()
    model = rf_spark_hard.fit(train_x5)
    t1 = time.perf_counter()
    dt = t1 - t0
    spark_fit_times.append(dt)
    print(f"Run Spark {i+1}: {dt:.2f} s")

spark_fit_median = float(np.median(spark_fit_times))
print(f"\nTemps fit median Spark (Codespace): {spark_fit_median:.2f} s")

# Optionnel: eval rapide
pred = model.transform(test_x5)
acc_x5 = acc_eval.evaluate(pred)
print(f"Accuracy Spark x5: {acc_x5:.4f}")

26/03/20 11:32:23 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:32:58 WARN MemoryStore: Not enough space to cache rdd_158_12 in memory! (computed 57.2 MiB so far)
26/03/20 11:32:58 WARN BlockManager: Persisting block rdd_158_12 to disk instead.
26/03/20 11:32:58 WARN MemoryStore: Not enough space to cache rdd_158_13 in memory! (computed 53.0 MiB so far)
26/03/20 11:32:58 WARN BlockManager: Persisting block rdd_158_13 to disk instead.
26/03/20 11:32:58 WARN MemoryStore: Not enough space to cache rdd_158_12 in memory! (computed 57.2 MiB so far)
26/03/20 11:32:58 WARN MemoryStore: Not enough space to cache rdd_158_13 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:07 WARN MemoryStore: Not enough space to cache rdd_158_17 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:07 WARN BlockManager: Persisting block rdd_158_17 to disk instead.
26/03/20 11:33:08 WARN MemoryStore: Not enough space to cache rdd_158_17 in memory! (computed 53.0 MiB so far)
26

Taille train Spark x5 : 280246


26/03/20 11:33:30 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:33:34 WARN MemoryStore: Not enough space to cache rdd_158_9 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:37 WARN MemoryStore: Not enough space to cache rdd_158_17 in memory! (computed 53.0 MiB so far)


Taille test  Spark x5 : 69754


26/03/20 11:33:41 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:33:47 WARN MemoryStore: Not enough space to cache rdd_158_12 in memory! (computed 57.2 MiB so far)
26/03/20 11:33:47 WARN MemoryStore: Not enough space to cache rdd_158_13 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:49 WARN MemoryStore: Not enough space to cache rdd_158_17 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:52 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:33:53 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:33:57 WARN MemoryStore: Not enough space to cache rdd_158_5 in memory! (computed 53.0 MiB so far)
26/03/20 11:33:59 WARN MemoryStore: Not enough space to cache rdd_158_9 in memory! (computed 53.0 MiB so far)
26/03/20 11:34:01 WARN MemoryStore: Not enough space to cache rdd_158_12 in memory! (computed 57.2 MiB so far)
26/03/20 11:34:04 WARN MemoryStore: Not enough space to cache rdd_158_17 in me

...

Run Spark 2: 88.05 s


26/03/20 11:36:48 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:36:53 WARN MemoryStore: Not enough space to cache rdd_158_12 in memory! (computed 57.2 MiB so far)
26/03/20 11:36:53 WARN MemoryStore: Not enough space to cache rdd_158_13 in memory! (computed 53.0 MiB so far)
26/03/20 11:36:55 WARN MemoryStore: Not enough space to cache rdd_158_17 in memory! (computed 53.0 MiB so far)
26/03/20 11:36:58 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:36:59 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/03/20 11:37:02 WARN MemoryStore: Not enough space to cache rdd_158_5 in memory! (computed 53.0 MiB so far)
26/03/20 11:37:05 WARN MemoryStore: Not enough space to cache rdd_158_9 in memory! (computed 53.0 MiB so far)
26/03/20 11:37:07 WARN MemoryStore: Not enough space to cache rdd_158_13 in memory! (computed 53.0 MiB so far)
26/03/20 11:37:09 WARN MemoryStore: Not enough space to cache rdd_158_17 in me

Run Spark 3: 90.32 s

Temps fit median Spark (Codespace): 90.32 s


NameError: name 'acc_eval' is not defined

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55536)
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/local/lib/python3.10/site-packages/pyspark/accumulators.py", line 300, in handle
    poll(authenticate_and_accum_updates)
  File "/usr/local/lib/python3.10/site-packages/pyspark/accumulators.py", line 272, in poll
    if self.rfile in r and func():
  File "/usr/local/lib/python3.10/site-packages/pyspark/accumulators.py", line 293, in 

## 8. Exercice  Benchmark local scikit-learn (meme protocole)

Objectif : refaire localement la meme experience que la section 9.

Consignes (local sklearn) :
1. Charger MNIST (telechargement automatique si absent).
2. Dupliquer le dataset **5 fois**.
3. Split train/test.
4. Entrainer un Random Forest exigeant (`n_estimators=300`, `max_depth=20`).
5. Mesurer uniquement le temps de `fit()`.
6. Faire 3 runs et garder la mediane.

Comparaison finale :
- comparer `temps fit median Spark (section 9)` et `temps fit median sklearn (section 10)`.
- conclusion indicative (environnements differents).

In [ ]:
# Exercice local sklearn: duplication x5 + Random Forest exigeant
# Cette cellule est autonome: telecharge MNIST si necessaire

import os
import time
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

url = "https://www.math.univ-toulouse.fr/~besse/Wikistat/data/"
local_data_dir = "Data/mnist"
os.makedirs(local_data_dir, exist_ok=True)

for filename in ["mnist_train.csv", "mnist_test.csv"]:
    target_path = os.path.join(local_data_dir, filename)
    if not os.path.exists(target_path):
        print(f"Telechargement de {filename}...")
        urllib.request.urlretrieve(url + filename, target_path)
    else:
        print(f"Fichier deja present : {target_path}")

train_csv = os.path.join(local_data_dir, "mnist_train.csv")
test_csv = os.path.join(local_data_dir, "mnist_test.csv")

# Chargement (hors chrono)
df_train = pd.read_csv(train_csv, header=None)
df_test = pd.read_csv(test_csv, header=None)
df = pd.concat([df_train, df_test], ignore_index=True)

# Dupliquer x5 (meme protocole que Spark)
df_x5 = pd.concat([df] * 5, ignore_index=True)

# Si la machine locale est trop limitee, vous pouvez commenter la ligne precedente
# et decommenter la suivante pour un mode degrade:
# df_x5 = pd.concat([df] * 3, ignore_index=True)

y = df_x5.iloc[:, 0].astype(np.uint8).to_numpy()
X = df_x5.iloc[:, 1:].astype(np.float32).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Taille train sklearn x5 :", len(X_train))
print("Taille test  sklearn x5 :", len(X_test))

# Random Forest exigeant
rf_sk_hard = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=2,
)

# Mesurer fit uniquement (3 runs)
sk_fit_times = []
for i in range(3):
    t0 = time.perf_counter()
    rf_sk_hard.fit(X_train, y_train)
    t1 = time.perf_counter()
    dt = t1 - t0
    sk_fit_times.append(dt)
    print(f"Run sklearn {i+1}: {dt:.2f} s")

sk_fit_median = float(np.median(sk_fit_times))
print(f"\nTemps fit median sklearn (local): {sk_fit_median:.2f} s")

acc_local = accuracy_score(y_test, rf_sk_hard.predict(X_test))
print(f"Accuracy sklearn x5: {acc_local:.4f}")

print("\nComparaison a faire avec la section 9 (Spark Codespace).")